#  Validar y medir un modelo

**Analítica de datos aplicada a la Ingeniería Civil**

**Caso:** resistencia a compresión del hormigón (1030 mezclas ensayadas · dataset de Yeh, UCI Machine Learning Repository)

> 🎯 **Mensaje clave:** elegir un modelo es elegir su flexibilidad, y eso se decide **validando**, no mirando el ajuste.

### Cómo trabajar con este notebook

- Ejecuta las celdas **en orden**, de arriba abajo (`Shift + Enter`).
- Las celdas marcadas con **✏️** tienen un hueco `___` que debes rellenar. Si no lo rellenas, la celda dará error.
- Los bloques **💡 Preguntas** son para pensar y comentar en clase: lo importante es **interpretar**, no escribir código.

| Bloque | Qué hacemos en este notebook |
|:--:|---|
| 1 | Cargar los datos y apartar la prueba |
| 2 | Validación simple: ¿depende del reparto? |
| 3 | Validación cruzada: elegir K y el grado del polinomio |
| 4 | Escalar las variables: pipeline escalado + KNN |
| 5 | La función común `evaluar_modelo()` y la tabla de modelos |
| 6 | Experimento: añadir variables de ruido |
| 7 | *Ampliación (si sobra tiempo):* validar no protege de extrapolar |

### Imports y configuración básica

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, KFold, cross_val_score, cross_validate, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Colores que usaremos en todo el bloque
AZUL = "#4C72B0"      # datos de entrenamiento
NARANJA = "#DD8452"   # datos de prueba / validación
ROJO = "#C44E52"      # modelo

plt.rcParams["font.size"] = 12


def estilo(ax):
    """Estética común de las gráficas: sin bordes arriba y a la derecha, rejilla suave."""
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(alpha=0.25)


def rmse(y_real, y_pred):
    """Raíz del error cuadrático medio, en las unidades de la respuesta (MPa)."""
    return np.sqrt(mean_squared_error(y_real, y_pred))

---
## 1. Cargar los datos y apartar la prueba

Mismo fichero, mismos nombres de variables y **mismo reparto** que hasta ahora: el 20 % de las mezclas se aparta como **prueba** y **no se vuelve a tocar** hasta el final del notebook.

In [ ]:
# Si trabajas en local, cambia la ruta por la de tu ordenador
ruta = "/workspaces/AnaliticaDeDatos/datos/clases/clase_1_analisis_exploratorio_de_datos/Concrete_Data.xls"
concrete_data = pd.read_excel(ruta, index_col=False)

concrete_data.columns = ["cemento", "escoria", "cenizas", "agua", "superplastificante",
                         "arido_grueso", "arido_fino", "edad", "resistencia"]

variables = ["cemento", "escoria", "cenizas", "agua", "superplastificante",
             "arido_grueso", "arido_fino", "edad"]
X = concrete_data[variables]
y = concrete_data["resistencia"]

print("Dimensiones del dataset:", concrete_data.shape)
print(f"Desviación típica de la resistencia: {y.std():.1f} MPa")

✏️ **Rellena el hueco:** usa la **misma semilla** que hasta ahora (`42`), para que la prueba sean las mismas 206 mezclas.

In [ ]:
X_ent, X_pru, y_ent, y_pru = train_test_split(X, y, test_size=0.2, random_state=___)   # ✏️

print("Entrenamiento:", len(X_ent), "mezclas")
print("Prueba:       ", len(X_pru), "mezclas  ← no las tocamos hasta el final")

> ### 💡 Preguntas
>
> 1) ¿Por qué nos interesa usar siempre la misma semilla (`random_state`)?
>
> 2) La desviación típica de la resistencia es de unos 17 MPa. ¿Qué error tendría un «modelo» que predijese siempre la media?

---
## 2. Validación simple: ¿depende del reparto?

Hasta ahora elegíamos K **mirando la prueba**. Eso es hacer trampa: la prueba pasa a formar parte de la decisión. La alternativa más sencilla es la **validación simple**: partir el **entrenamiento** en dos mitades, entrenar con una y medir el error en la otra (la de **validación**).

### Formulación

Con un hiperparámetro $\lambda$ fijo (aquí, $K$), el modelo $\hat f$ se entrena con la mitad de entrenamiento y se mide su error en las $n_{\text{val}}$ mezclas de validación:

$$E_{\text{val}}(\lambda) = \sqrt{\frac{1}{n_{\text{val}}}\sum_{i \in \text{val}} \left(y_i - \hat f(x_i;\lambda)\right)^2} \qquad \text{(RMSE, en MPa)}$$

y se elige $\lambda^* = \arg\min_\lambda E_{\text{val}}(\lambda)$.

> En este bloque usamos KNN con las variables **escaladas** (`knn_escalado`). Qué es escalar y por qué hace falta lo vemos con calma en el bloque 4; de momento, úsalo como un modelo más.

In [ ]:
def knn_escalado(k):
    """KNN con las variables estandarizadas (lo explicamos en el bloque 4)."""
    return make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=k))


Ks = [1, 2, 3, 4, 5, 7, 10, 15, 20, 30]

# Un único reparto 50/50 del ENTRENAMIENTO (la prueba sigue apartada)
X_a, X_v, y_a, y_v = train_test_split(X_ent, y_ent, test_size=0.5, random_state=0)
err_val = [rmse(y_v, knn_escalado(k).fit(X_a, y_a).predict(X_v)) for k in Ks]

pd.DataFrame({"K": Ks, "RMSE validación (MPa)": np.round(err_val, 2)}).T

✏️ **Rellena el hueco:** el K elegido es el de **menor error** de validación (`np.argmin` da la posición del mínimo).

In [ ]:
k_val = Ks[int(np.___(err_val))]   # ✏️
print(f"Con este reparto elegiríamos K = {k_val} (RMSE de validación = {min(err_val):.2f} MPa)")

Ahora repetimos **lo mismo con 10 repartos distintos** (10 semillas). Cada línea es un reparto; el punto marca el K que elegiría.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
axes[0].plot(Ks, err_val, marker="o", color=AZUL, linewidth=2)
axes[0].plot(k_val, min(err_val), "o", ms=14, mfc="none", mec=NARANJA, mew=2.5)
axes[0].set_title("Un reparto entrenamiento/validación")

colores = plt.cm.viridis(np.linspace(0.1, 0.9, 10))
k_elegidos, err_minimos = [], []
for semilla in range(10):
    X_a, X_v, y_a, y_v = train_test_split(X_ent, y_ent, test_size=0.5, random_state=semilla)
    err = [rmse(y_v, knn_escalado(k).fit(X_a, y_a).predict(X_v)) for k in Ks]
    i_min = int(np.argmin(err))
    k_elegidos.append(Ks[i_min]); err_minimos.append(err[i_min])
    axes[1].plot(Ks, err, color=colores[semilla], linewidth=1.8)
    axes[1].plot(Ks[i_min], err[i_min], "o", color=colores[semilla], ms=7)
axes[1].set_title("Diez repartos distintos (10 semillas)")

for ax in axes:
    ax.set_xlabel("K (número de vecinos)")
    estilo(ax)
axes[0].set_ylabel("RMSE de validación (MPa)")
plt.tight_layout()
plt.show()

print("K elegido con cada semilla:", k_elegidos)
print(f"Error del K elegido: de {min(err_minimos):.2f} a {max(err_minimos):.2f} MPa según el reparto")

> ### 💡 Preguntas
>
> 1) ¿Cambia mucho el K elegido de un reparto a otro? ¿Y el error que estimamos?
>
> 2) Si tuvieras que escribir en un informe «el modelo se equivoca X MPa», ¿qué número pondrías?
>
> 3) ¿Con cuántas mezclas se entrena cada modelo en este bloque? ¿Es un problema?

---
## 3. Validación cruzada

La **validación cruzada k-fold** divide el entrenamiento en $k$ pliegues. Cada pliegue hace **una vez** de validación mientras los otros $k-1$ entrenan, y se promedian los $k$ errores.

### Formulación

Si $\hat f^{(-j)}$ es el modelo entrenado **sin** el pliegue $j$ y $n_j$ el número de mezclas de ese pliegue:

$$E_j(\lambda) = \sqrt{\frac{1}{n_j}\sum_{i \in \text{pliegue } j} \left(y_i - \hat f^{(-j)}(x_i;\lambda)\right)^2}, \qquad \text{CV}(\lambda) = \frac{1}{k}\sum_{j=1}^{k} E_j(\lambda), \qquad \lambda^* = \arg\min_{\lambda} \text{CV}(\lambda)$$

(ISLP, sección 5.1.3; allí se promedia el MSE, aquí promediamos el RMSE de cada pliegue, que es lo que hace scikit-learn).

### 3.1. Una validación cruzada «a mano»

✏️ **Rellena el hueco:** queremos **5 pliegues**.

In [ ]:
cv = KFold(n_splits=___, shuffle=True, random_state=42)   # ✏️  mismos pliegues para todo el notebook

puntuaciones = cross_val_score(knn_escalado(5), X_ent, y_ent, cv=cv,
                               scoring="neg_root_mean_squared_error")
errores = -puntuaciones   # scikit-learn devuelve el error con signo negativo (siempre maximiza)

print("RMSE en cada pliegue (MPa):", np.round(errores, 2))
print(f"Validación cruzada: RMSE = {errores.mean():.2f} ± {errores.std():.2f} MPa")

> ### 💡 Preguntas
>
> 1) ¿Cuántas mezclas hay en cada pliegue de validación? ¿Con cuántas se entrena cada vez?
>
> 2) ¿Por qué aparece un signo menos en `errores = -puntuaciones`?

### 3.2. Elegir K con validación cruzada

Repetimos la validación cruzada para K = 1, …, 50 y guardamos la **media** y la **desviación** del error entre los 5 pliegues.

✏️ **Rellena el hueco:** queremos K desde 1 hasta 50 (recuerda: `range(1, N)` va de 1 a N − 1).

In [ ]:
K_values = list(range(1, ___))   # ✏️

cv_media, cv_desv = [], []
for k in K_values:
    e = -cross_val_score(knn_escalado(k), X_ent, y_ent, cv=cv, scoring="neg_root_mean_squared_error")
    cv_media.append(e.mean())
    cv_desv.append(e.std())
cv_media, cv_desv = np.array(cv_media), np.array(cv_desv)

i_min = int(np.argmin(cv_media))
k_cv = K_values[i_min]

# Regla práctica: el K más simple (más grande) cuyo error está a menos de un error estándar del mínimo
limite = cv_media[i_min] + cv_desv[i_min] / np.sqrt(5)
k_simple = max(k for k, m in zip(K_values, cv_media) if m <= limite)

fig, ax = plt.subplots(figsize=(9, 5))
ax.fill_between(K_values, cv_media - cv_desv, cv_media + cv_desv, color=AZUL, alpha=0.15,
                label="± 1 desviación entre pliegues")
ax.plot(K_values, cv_media, color=AZUL, linewidth=2.2, label="RMSE medio de validación cruzada")
ax.axhline(limite, color="grey", linestyle=":", linewidth=1)
ax.plot(k_cv, cv_media[i_min], "o", color=NARANJA, ms=10, label=f"Mínimo: K = {k_cv}")
ax.plot(k_simple, cv_media[k_simple - 1], "s", color=ROJO, ms=9, label=f"Más simple casi empatado: K = {k_simple}")
ax.set_xlabel("K (número de vecinos)   ← más flexible · más rígido →")
ax.set_ylabel("RMSE (MPa)")
ax.set_title("KNN escalado: elegir K con validación cruzada de 5 pliegues")
ax.legend(frameon=False)
estilo(ax)
plt.show()

print(f"K con menor error: {k_cv} → RMSE = {cv_media[i_min]:.2f} ± {cv_desv[i_min]:.2f} MPa")
print(f"K más simple casi empatado: {k_simple} → RMSE = {cv_media[k_simple - 1]:.2f} MPa")

> ### 💡 Preguntas
>
> 1) ¿Dónde está el sobreajuste en esta gráfica? ¿Y el infraajuste?
>
> 2) La diferencia de error entre el K mínimo y el «más simple casi empatado», ¿es grande comparada con la banda sombreada?
>
> 3) Esta curva en U, ¿con qué datos la hemos dibujado? Compárala con lo que hacíamos hasta ahora.

### 3.3. Elegir el grado del polinomio sobre la edad

Lo mismo sirve para cualquier hiperparámetro. Elegimos el **grado del polinomio** usando solo la edad.

✏️ **Rellena el hueco:** la función de validación cruzada que devuelve una puntuación por pliegue es `cross_val_score`.

In [ ]:
def modelo_polinomico(grado):
    """Polinomio de grado `grado` de las variables de entrada (escalamos antes de elevar a potencias)."""
    return make_pipeline(StandardScaler(), PolynomialFeatures(grado), LinearRegression())


grados = list(range(1, 9))
cv_grado = [-___(modelo_polinomico(g), X_ent[["edad"]], y_ent, cv=cv,   # ✏️
                               scoring="neg_root_mean_squared_error").mean() for g in grados]
grado_cv = grados[int(np.argmin(cv_grado))]

pd.DataFrame({"Grado": grados, "RMSE validación cruzada (MPa)": np.round(cv_grado, 2)}).T

In [ ]:
print(f"Grado elegido por validación cruzada (solo la edad): {grado_cv}")

# Comparación rápida: polinomios con las 8 variables
for g in [1, 2, 3, 4]:
    e = -cross_val_score(modelo_polinomico(g), X_ent, y_ent, cv=cv, scoring="neg_root_mean_squared_error").mean()
    print(f"8 variables, grado {g}: RMSE de validación cruzada = {e:8.1f} MPa")

> ### 💡 Preguntas
>
> 1) Con solo la edad, ¿cuánto baja el error al pasar de grado 1 al grado elegido? ¿Es un buen modelo comparado con los 17 MPa de «predecir la media»?
>
> 2) Con las 8 variables, ¿qué ocurre en grado 4? Recuerda cuántos parámetros tiene.

### 3.4. Atajo: `GridSearchCV`

`GridSearchCV` hace de una vez lo que hemos programado en el bloque 3.2: prueba toda la rejilla con validación cruzada, elige el mejor y **lo reentrena con todo el entrenamiento**.

En un pipeline, los hiperparámetros se nombran como `paso__parámetro` (dos guiones bajos).

✏️ **Rellena el hueco:** el nombre es el del paso (`kneighborsregressor`) + dos guiones bajos + el del parámetro de KNN que guarda el número de vecinos (`n_neighbors`).

In [ ]:
busqueda = GridSearchCV(knn_escalado(5),
                        {"___": K_values},   # ✏️
                        cv=cv, scoring="neg_root_mean_squared_error")
busqueda.fit(X_ent, y_ent)

print("K elegido por GridSearchCV:", busqueda.best_params_["kneighborsregressor__n_neighbors"])
print(f"RMSE de validación cruzada del mejor: {-busqueda.best_score_:.2f} MPa")

---
## 4. Escalar las variables: pipeline escalado + KNN

KNN decide qué mezclas se parecen con una **distancia euclídea**:

$$d(\mathbf{x}, \mathbf{x}') = \sqrt{\sum_{j=1}^{p} \left(x_j - x'_j\right)^2}$$

Cada variable entra con sus **unidades**: una diferencia de 10 kg/m³ de cemento pesa lo mismo que 10 kg/m³ de superplastificante o que 10 días de edad. **Estandarizar** pone todas las variables en la misma escala:

$$z_j = \frac{x_j - \bar x_j}{s_j} \qquad \text{(media 0 y desviación 1, calculadas solo con el entrenamiento)}$$

Con un **pipeline** (`make_pipeline(StandardScaler(), KNeighborsRegressor())`), en cada pliegue de la validación cruzada el escalador aprende $\bar x_j$ y $s_j$ **solo con las mezclas de entrenamiento de ese pliegue**: así no hay fuga de datos.

### 4.1. ¿Cuánto cambia KNN al escalar?

In [ ]:
def rmse_cv(modelo, X_datos):
    """RMSE medio de validación cruzada (mismos 5 pliegues de siempre)."""
    return -cross_val_score(modelo, X_datos, y_ent, cv=cv, scoring="neg_root_mean_squared_error").mean()


print(f"KNN (K = 5) sin escalar : RMSE = {rmse_cv(KNeighborsRegressor(5), X_ent):.2f} MPa")
print(f"KNN (K = 5) escalado    : RMSE = {rmse_cv(knn_escalado(5), X_ent):.2f} MPa")

### 4.2. Experimento: medir la edad en horas

Cambiamos **solo las unidades** de la edad (días → horas). La información es exactamente la misma.

✏️ **Rellena el hueco:** un día tiene 24 horas.

In [ ]:
X_ent_horas = X_ent.copy()
X_ent_horas["edad"] = X_ent_horas["edad"] * ___   # ✏️

print(f"Edad en horas, sin escalar : RMSE = {rmse_cv(KNeighborsRegressor(5), X_ent_horas):.2f} MPa")
print(f"Edad en horas, escalado    : RMSE = {rmse_cv(knn_escalado(5), X_ent_horas):.2f} MPa")

> ### 💡 Preguntas
>
> 1) ¿Mejora mucho KNN al escalar con estos datos? ¿Por qué creéis que mejora tan poco?
>
> 2) Sin escalar, cambiar la edad a horas **cambia el modelo**. ¿Por qué? ¿Y por qué escalado da lo mismo en días que en horas?
>
> 3) Sin escalar y en horas, KNN acierta **más**. ¿Significa que hay que medir la edad en horas? ¿Qué nos está diciendo este resultado?

---
## 5. La función común `evaluar_modelo()` y la tabla de modelos

A partir de ahora **todos los modelos** se evalúan igual: validación cruzada de 5 pliegues (siempre los mismos), sobre el entrenamiento, con tres métricas.

### Formulación

Con $n$ mezclas evaluadas (las de validación de cada pliegue):

$$\text{MAE} = \frac{1}{n}\sum_{i=1}^{n}\left|y_i - \hat y_i\right| \qquad \text{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n}\left(y_i - \hat y_i\right)^2} \qquad R^2 = 1 - \frac{\sum_i (y_i - \hat y_i)^2}{\sum_i (y_i - \bar y)^2}$$

MAE y RMSE se leen en **MPa**; el RMSE penaliza más los errores grandes y siempre es $\geq$ MAE.

✏️ **Rellena el hueco:** como scikit-learn da los errores en negativo, hay que **cambiarles el signo** multiplicando por `-1` (MAE y RMSE).

In [ ]:
tabla_modelos = []


def evaluar_modelo(nombre, modelo, X, y, comentario=""):
    """Validación cruzada de 5 pliegues (siempre los mismos) y una fila nueva en la tabla de modelos."""
    res = cross_validate(modelo, X, y, cv=cv,
                         scoring=["neg_mean_absolute_error", "neg_root_mean_squared_error", "r2"])
    fila = {"Modelo": nombre,
            "MAE (MPa)": round(___ * res["test_neg_mean_absolute_error"].mean(), 2),        # ✏️
            "RMSE (MPa)": round(___ * res["test_neg_root_mean_squared_error"].mean(), 2),   # ✏️
            "R²": round(res["test_r2"].mean(), 3),
            "Comentario": comentario}
    tabla_modelos.append(fila)
    return fila


evaluar_modelo("Regresión lineal", LinearRegression(), X_ent, y_ent, comentario="8 variables")
evaluar_modelo(f"Polinomio sobre la edad (grado {grado_cv})", modelo_polinomico(grado_cv), X_ent[["edad"]], y_ent,
               comentario="solo la edad · grado elegido por VC")

Para la fila de **KNN sin escalar** elegimos también su K por validación cruzada:

In [ ]:
busqueda_sin = GridSearchCV(KNeighborsRegressor(), {"n_neighbors": K_values}, cv=cv,
                            scoring="neg_root_mean_squared_error").fit(X_ent, y_ent)
k_sin = busqueda_sin.best_params_["n_neighbors"]

evaluar_modelo(f"KNN sin escalar (K = {k_sin})", KNeighborsRegressor(k_sin), X_ent, y_ent, comentario="K elegido por VC")
evaluar_modelo(f"KNN escalado (K = {k_cv})", knn_escalado(k_cv), X_ent, y_ent, comentario="K elegido por VC")

tabla = pd.DataFrame(tabla_modelos)
tabla

> ### 💡 Preguntas
>
> 1) ¿Qué modelo es el mejor hoy? ¿Cuántos MPa se equivoca? Compáralo con los 17 MPa de predecir la media y con los 5 MPa que separan dos clases resistentes (HA-25, HA-30…).
>
> 2) En todos los modelos, ¿el RMSE es mayor que el MAE? ¿Qué indica que lo sea bastante?
>
> 3) ¿Podríamos haber comparado estos cuatro modelos con el AIC?

### El último paso: evaluar **una sola vez** en la prueba

Elegimos el mejor modelo de la tabla, lo reentrenamos con **todo** el entrenamiento y lo evaluamos **una vez** con las 206 mezclas de prueba que apartamos al principio. Ese es el número que iría al informe.

In [ ]:
modelo_final = knn_escalado(k_cv).fit(X_ent, y_ent)
pred_pru = modelo_final.predict(X_pru)

print(f"Modelo final: KNN escalado (K = {k_cv})")
print(f"  Prueba → MAE = {mean_absolute_error(y_pru, pred_pru):.2f} MPa | "
      f"RMSE = {rmse(y_pru, pred_pru):.2f} MPa | R² = {r2_score(y_pru, pred_pru):.3f}")

fig, ax = plt.subplots(figsize=(6, 6))
e = rmse(y_pru, pred_pru)
ax.fill_between([0, 85], [-e, 85 - e], [e, 85 + e], color=AZUL, alpha=0.12, label=f"± RMSE = {e:.1f} MPa")
ax.scatter(y_pru, pred_pru, color=NARANJA, edgecolor="white", s=40, label="Mezclas de prueba")
ax.plot([0, 85], [0, 85], color=ROJO, linestyle="--", label="Ajuste perfecto")
ax.set_xlim(0, 85); ax.set_ylim(0, 85); ax.set_aspect("equal")
ax.set_xlabel("Resistencia observada (MPa)"); ax.set_ylabel("Resistencia predicha (MPa)")
ax.set_title("Modelo final evaluado en la prueba")
ax.legend(frameon=False, loc="upper left")
estilo(ax)
plt.show()

In [ ]:
tabla.to_csv("tabla_modelos_hormigon.csv", index=False)
print("Tabla guardada en tabla_modelos_hormigon.csv")

---
## 6. Experimento: añadir variables de ruido

¿Y si metemos «todas las variables por si acaso»? Añadimos columnas de **números aleatorios** (sin ninguna relación con la resistencia) y comparamos KNN escalado con la regresión lineal.

✏️ **Rellena el hueco:** prueba a añadir 0, 2, 5, 10, 20 y **40** variables de ruido.

In [ ]:
n_ruido = [0, 2, 5, 10, 20, ___]   # ✏️

rng = np.random.default_rng(0)
err_knn, err_lin = [], []
for p in n_ruido:
    ruido = pd.DataFrame(rng.uniform(0, 1, size=(len(X_ent), p)),
                         columns=[f"ruido_{i}" for i in range(p)], index=X_ent.index)
    X_ruido = pd.concat([X_ent, ruido], axis=1)
    err_knn.append(rmse_cv(knn_escalado(5), X_ruido))
    err_lin.append(rmse_cv(LinearRegression(), X_ruido))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(n_ruido, err_knn, marker="o", color=NARANJA, linewidth=2, label="KNN escalado (K = 5)")
ax.plot(n_ruido, err_lin, marker="s", color=AZUL, linewidth=2, label="Regresión lineal")
ax.set_xlabel("Nº de variables de ruido añadidas a las 8 del hormigón")
ax.set_ylabel("RMSE de validación cruzada (MPa)")
ax.set_title("Más variables no siempre es mejor")
ax.legend(frameon=False)
estilo(ax)
plt.show()

pd.DataFrame({"Variables de ruido": n_ruido, "KNN (MPa)": np.round(err_knn, 2),
              "Lineal (MPa)": np.round(err_lin, 2)}).T

> ### 💡 Preguntas
>
> 1) ¿Qué modelo sufre más al añadir variables inútiles? ¿Por qué?
>
> 2) Con 40 variables de ruido, ¿cuál de los dos elegirías?
>
> 3) ¿Qué harías en un caso real con muchas variables candidatas?

---
## 7. *Ampliación (si sobra tiempo):* validar no protege de extrapolar

Entrenamos solo con mezclas de **hasta 350 kg/m³ de cemento** y usamos el modelo con mezclas de **420 kg/m³ o más** (hormigones de alta resistencia, fuera del rango de entrenamiento).

In [ ]:
pobres = concrete_data["cemento"] <= 350
ricas = concrete_data["cemento"] >= 420
X_p, y_p = concrete_data.loc[pobres, variables], concrete_data.loc[pobres, "resistencia"]
X_r, y_r = concrete_data.loc[ricas, variables], concrete_data.loc[ricas, "resistencia"]

for nombre, modelo in [("Regresión lineal", LinearRegression()), ("KNN escalado (K = 5)", knn_escalado(5)),
                       ("Polinomio grado 3", modelo_polinomico(3))]:
    e_cv = -cross_val_score(modelo, X_p, y_p, cv=cv, scoring="neg_root_mean_squared_error").mean()
    modelo.fit(X_p, y_p)
    e_uso = rmse(y_r, modelo.predict(X_r))
    print(f"{nombre:22s} → validación cruzada (≤ 350 kg): {e_cv:5.1f} MPa | mezclas ≥ 420 kg: {e_uso:5.1f} MPa")

> ### 💡 Pregunta
>
> La validación cruzada prometía errores de 6–10 MPa. ¿Por qué no nos avisó de lo que iba a pasar con las mezclas ricas en cemento?

---
### 📌 Para llevarse

- **Tres conjuntos, tres papeles:** entrenamiento para ajustar, validación para elegir, prueba para informar (**una sola vez**).
- **Validación cruzada k-fold:** cada mezcla valida una vez; el error medio es mucho más estable que con un solo reparto. En la práctica, $k = 5$ o $10$.
- **Los hiperparámetros se eligen validando:** el de menor error, o el más simple que casi empata.
- **Escalar** hace que el peso de cada variable sea una decisión y no un accidente de las unidades; con un **pipeline**, sin fuga de datos.
- **Métricas en MPa:** MAE y RMSE se explican en las unidades del problema y se comparan con referencias (la media, las clases resistentes).
- **Más variables no siempre es mejor**, sobre todo para KNN.

**Próximamente:** aprender **sin variable respuesta** (PCA y clustering).